# Experiment with various Transformer based models

In [ ]:
from datetime import datetime
import copy
import os
import random
from typing import Any
import gc

from sklearn.preprocessing import LabelEncoder
import shap
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from mlflow.entities import Run
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import seaborn as sns
from datasets import DatasetDict
from transformers import (
    AutoTokenizer, AutoConfig, TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification, BatchEncoding, EvalPrediction
)
import torch
from transformers import Trainer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
import optuna

from ixbrl_ai.display import heading, display_wide
from ixbrl_ai.sample import DataSample
from ixbrl_ai.data import get_split

from torch.nn import CrossEntropyLoss
from sklearn.utils.class_weight import compute_class_weight


# 1. Config, setup mlflow, load data

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set random seed
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

device_type = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else device_type)
print(f"Device selected is {DEVICE}")

experiment_name = "sentence-transformers-compare"
mlflow.set_experiment(experiment_name)
mlflow.transformers.autolog()

# mpnet might be better for longer text, mini might be faster and do just as well
MODELS = ["roberta-base", "nlpaueb/sec-bert-base", "sentence-transformers/all-mpnet-base-v2", "sentence-transformers/all-MiniLM-L6-v2"]

In [ ]:
X = "canonical_description"
y = "label"
dataset_version = "v13"
dataset_name = f"data/canonicalized_split_{dataset_version}.parquet"
dataset_pl = pl.read_parquet(dataset_name)
subset = DataSample.sample_1_pct
train_pl, test_pl, holdout_pl = get_split(dataset_pl, subset)
sample_1_pct_sqrt_weight = dataset_pl.filter(pl.col("sample_1_pct_sqrt_weight"))
sample_10_pct_sqrt_weight = dataset_pl.filter(pl.col("sample_10_pct_sqrt_weight"))
test_5_pct_pl = dataset_pl.filter(pl.col("test_5_pct"))

rng = np.random.default_rng(seed=SEED)
idx_holdout_10k = rng.choice(dataset_pl.filter(pl.col("holdout")).select("row_id"), size=10000, replace=False)
idx_holdout_10k.flatten()
holdout_10k_pl = dataset_pl.filter(pl.col("row_id").is_in(idx_holdout_10k.flatten()))

datasets = {
    "train": train_pl,
    DataSample.sample_1_pct.label: dataset_pl.filter(pl.col(DataSample.sample_1_pct.label)),
    DataSample.sample_10_pct.label: dataset_pl.filter(pl.col(DataSample.sample_10_pct.label)),
    DataSample.sample_100_pct.label: dataset_pl.filter(pl.col(DataSample.sample_100_pct.label)),
    "sample_1_pct_sqrt_weight": sample_1_pct_sqrt_weight,
    "sample_10_pct_sqrt_weight": sample_10_pct_sqrt_weight,
    "test": test_pl,
    "test_5_pct": test_5_pct_pl,
    "holdout": holdout_pl,
    "holdout_10k": holdout_10k_pl,
}


# model_name = "sentence-transformers/all-mpnet-base-v2"

def tokenize_for_model(model_name: str, datasets: dict[str, pl.DataFrame], max_length: int=64) -> dict[str, Dataset]:
    """Tokenizes dataset using model

    Args:
        model_name (str): Model name
        datasets (dict[str, pl.DataFrame]): The datasets to tokenize
        max_length (int, optional): Max token length. Defaults to 64.

    Returns:
        dict[str, Dataset]: Tokenized datasets
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch: dict[str, list[Any]]) -> BatchEncoding:
        return tokenizer(batch[X], padding="max_length", truncation=True, max_length=max_length)

    def encode(data_pl: pl.DataFrame) -> Dataset:
        dataset = Dataset.from_polars(data_pl.select(X, y))
        dataset_encoded = dataset.map(tokenize, batched=True)
        dataset_encoded = dataset_encoded.remove_columns([X])
        dataset_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
        return dataset_encoded

    return {key: encode(value) for key, value in datasets.items()}




In [ ]:
datasets_encoded = {model: tokenize_for_model(model, datasets) for model in MODELS}

In [ ]:
# Characters
dataset_pl.select(pl.col(X).str.len_chars().max()).pipe(display)
# Words
dataset_pl.select(pl.col(X).str.split(" ").list.len().max())

In [ ]:
print(training_args.per_device_train_batch_size)

In [ ]:
all_lengths = []
for model_name in ["nlpaueb/sec-bert-base", "roberta-base", "sentence-transformers/all-mpnet-base-v2"]:
    for split in ["train", "test", "holdout"]:
        all_lengths += [sum(mask) for mask in datasets_encoded[model_name][split]["attention_mask"]]

print(max(all_lengths))

max_length was 32, so setting it to 32 should be fine. If there is a very rare occasion where there are more than 32 tokens, it's fine to truncate it. 

In [ ]:
datasets_encoded = {model: tokenize_for_model(model_name=model, datasets=datasets, max_length=32) for model in MODELS}

In [ ]:
# Do this over the full dataset, since you need continuous label's, using train messes it up completely
unique_pl = dataset_pl.select("canonical_label", "label").unique().sort("label")
id2label = dict(zip(unique_pl["label"], unique_pl["canonical_label"]))
label2id = dict(zip(unique_pl["canonical_label"], unique_pl["label"]))
num_labels=unique_pl.height

## 1.1 Functions

In [ ]:
def compute_metrics(eval_pred: EvalPrediction) -> dict[str, Any]:
    """Computes macro metrics

    Args:
        eval_pred (EvalPrediction): Predictions

    Returns:
        dict[str, float]: Metrics
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro"
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1_macro": f1,
    }


def load_best_trainer(experiment_name: str, run_name: str) -> tuple[Trainer, str, dict]:
    """Loads the best trainer from run and returns original model name and args

    Args:
        run_name (str): Run rame

    Returns:
        tuple[Trainer, str]: Return best trainer, model name and args
    """
    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    parent_runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
    )
    parent_run = parent_runs[0]

    child_runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'",
        order_by=["metrics.eval_f1_macro DESC"],
    )
    best_run = child_runs[0]
    model_path = f"/Volumes/WDElement/ML/EPA/bert/{run_name}/{best_run.info.run_name}"

    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    training_args = torch.load(f"{model_path}/training_args.bin", weights_only=False)
    original_args = copy.deepcopy(training_args)
    training_args.eval_strategy = "no"
    initial_model = best_run.data.params["_name_or_path"]
    print(f"model_path: {model_path}")
    print(f"model_type: {initial_model}")
    print(f"model_args {training_args}")
    return Trainer(model=model, args=training_args), initial_model, original_args


def get_run_details(
    experiment_name: str, run_name: str, index: int = 0
) -> pl.DataFrame:
    """Puts run details into a dataframe

    Args:
        experiment_name (str): Experiment name
        run_name (str): Run rame

    Returns:
        pl.DataFrame: Dataframe with metrics and params
    """

    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    parent_runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
    )
    parent_run = parent_runs[index]

    child_runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'",
        order_by=["metrics.eval_f1_macro DESC"],
    )

    runs_data = []
    for run in child_runs:
        run_data = {
            "model": run.data.params["_name_or_path"],
            "run": run.info.run_name,
            **run.data.metrics,
            **run.data.params,
        }
        runs_data.append(run_data)

    return pl.DataFrame(runs_data).with_row_index("rank")

def test_predictions_batch(
    trainer: Trainer,
    model_name: str,
    datasets_encoded: dict[str, dict[str, Dataset]],
    test_name: str,
    batch_size: int = 20000,
) -> dict[str, float]:
    """Prints prediction metrics using batch prediction

    Args:
        trainer (Trainer): HuggingFace Trainer
        model_name (str): Model name
        datasets_encoded (dict[str, dict[str, Dataset]]): Nested dict with the datasets
        test_name (str): Key used for dataset to test against
        batch_size (int): Batch size for prediction
    """
    dataset_test = datasets_encoded[model_name][test_name]
    predictions = []
    labels = []
    for i in range(0, len(dataset_test), batch_size):
        batch = dataset_test.select(range(i, min(i + batch_size, len(dataset_test))))
        prediction_options = trainer.predict(batch)
        predictions.extend(np.argmax(prediction_options.predictions, axis=1))
        labels.extend(prediction_options.label_ids)
        del prediction_options
        gc.collect()
        torch.mps.empty_cache()

    precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average="macro"
        )
    metrics = {
            "accuracy": accuracy_score(labels, predictions),
            "precision": precision,
            "recall": recall,
            "f1_macro": f1,
        }

    for key, value in metrics.items():
        print(f"{key}: {value}")

    


    return metrics


def test_predictions(
    trainer: Trainer,
    model_name: str,
    datasets_encoded: dict[str, dict[str, Dataset]],
    test_name: str,
) -> dict[str, float]:
    """Prints prediction metrics

    Args:
        trainer (Trainer): HuggingFace Trainer
        model_name (str): Model name
        datasets_encoded (dict[str, dict[str, Dataset]]): Nested dict with the datasets
        test_name (str): Key used for dataset to test against
    """
    test_encoded = datasets_encoded[model_name][test_name]
    prediction_options = trainer.predict(test_encoded)
    metrics = compute_metrics(
        EvalPrediction(
            predictions=prediction_options.predictions,
            label_ids=prediction_options.label_ids,
        )
    )

    for key, value in metrics.items():
        print(f"{key}: {value}")

    del prediction_options
    gc.collect()
    torch.mps.empty_cache()

    return metrics



def train_model(
    model_name: str,
    training_args: dict,
    train_subset: str,
    run_prefix: str,
    num_train_epochs: int | None = None,
) -> Trainer:

    run_name = f"{run_prefix}_{train_subset}"
    output_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_name}/trial_0"
    
    with mlflow.start_run(run_name=run_name) as parent_run:
        with mlflow.start_run(run_name="trial_0", nested=True):
            training_args.output_dir = output_dir

            if num_train_epochs is not None:
                training_args.num_train_epochs = num_train_epochs

            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=num_labels,
                id2label=id2label,
                label2id=label2id,
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=datasets_encoded[model_name][train_subset],
                eval_dataset=datasets_encoded[model_name]["test_5_pct"],
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
            )

            mlflow.log_params(training_args.to_dict())
            mlflow.log_param("train_subset", train_subset)
            start_time = datetime.now()
            trainer.train()
            end_time = datetime.now()
            best_metric = trainer.state.best_metric
            mlflow.log_metric("best_metric", best_metric)
            duration = (end_time - start_time).total_seconds()
            print(f"duration: {duration}s")
            mlflow.log_metric("train_time", duration)
            trainer.save_model(output_dir)
            AutoTokenizer.from_pretrained(model_name).save_pretrained(output_dir)
    


    return trainer




def load_trainer(
    model_name: str, run_prefix: str, train_subset: str
) -> Trainer:
    """Loads best model from run and tests it

    Args:
        model_name (str): Model name
        run_prefix (str): Run prefix
        train_subset (str): Training subset used

    Returns:
        Trainer: Loaded trainer
    """
    model_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_prefix}_{train_subset}/trial_0"
 
    trainer = Trainer(
        model=AutoModelForSequenceClassification.from_pretrained(model_dir),
        args=TrainingArguments(per_device_eval_batch_size=128, eval_accumulation_steps=8),
        )
    return trainer


In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro"
    )
metrics = {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1_macro": f1,
    }
print(metrics)

In [ ]:
datasets_encoded[model_name]["holdout"].shuffle().batch(1000)

In [ ]:
train_subset = DataSample.sample_100_pct.label
datasets_encoded[model_name][train_subset]

In [ ]:
train_subset = DataSample.sample_10_pct.label
datasets_encoded[model_name][train_subset]

In [ ]:
print(len(datasets_encoded[model_name]["train"]))
print(datasets_encoded[model_name]["train"].shape)

In [ ]:
test_encoded = datasets_encoded[model_name]["sample_100_pct"]

In [ ]:
trainer.train_dataset


In [ ]:
print(training_args.num_train_epochs)
print(training_args.gradient_accumulation_steps)

In [ ]:
print(len(trainer.train_dataset))

In [ ]:
best_run.data.metrics["eval_f1_macro"]

In [ ]:
best_run.data.params

# 2. Optuna compare different models and hyperparameters

In [ ]:
def objective(trial: optuna.Trial):

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        model_name = trial.suggest_categorical("model", MODELS)
        # dropout=trial.suggest_float("dropout", 0.1, 0.4)
        
        config = AutoConfig.from_pretrained(
            model_name, 
            num_labels=num_labels, 
            id2label=id2label, 
            label2id=label2id,
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, 
            config=config,
            )
        model = model.to(DEVICE)

        output_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_name}/trial_{trial.number}"
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=trial.suggest_categorical("per_device_train_batch_size", [16, 32, 64]),
            per_device_eval_batch_size=64,
            warmup_ratio=trial.suggest_float("warmup_ratio", 0.0, 0.3), 
            learning_rate=trial.suggest_float("learning_rate", 1e-6, 1e-4, log=True),
            weight_decay=trial.suggest_float("weight_decay", 0.0, 0.1),
            lr_scheduler_type=trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
            num_train_epochs=6,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            logging_steps=100,
            logging_dir="logs",
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1_macro", 
            greater_is_better=True,
            fp16=False,
            bf16=False,
            dataloader_pin_memory=False,
        )

        # trainer = WeightedTrainer(
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=datasets_encoded[model_name]["train"],
            eval_dataset=datasets_encoded[model_name]["test_5_pct"],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        mlflow.log_params(trial.params)
        start_time = datetime.now()
        trainer.train()
        end_time = datetime.now()
        best_metric = trainer.state.best_metric
        mlflow.log_metric("best_metric", best_metric)
        # mlflow.log_param("model_name", model_name)
        # mlflow.log_param("trial_number", trial.number)
        # mlflow.log_param("dropout", dropout)
        mlflow.log_metric("train_time", (end_time-start_time).total_seconds())
        trainer.save_model(output_dir)
        AutoTokenizer.from_pretrained(model_name).save_pretrained(output_dir)

        del model
        del trainer

        gc.collect()
        torch.mps.empty_cache()

    return best_metric



In [ ]:
run_name="bert_models_compared_no_weighting_fixed"

In [ ]:
with mlflow.start_run(run_name=run_name) as parent_run:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        # Warmup steps is warmup epochs
        pruner=optuna.pruners.MedianPruner(n_startup_trials=12, n_warmup_steps=2),
        study_name=run_name,
    )
    global_start_time = datetime.now()
    study.optimize(objective, n_trials=80)
    global_end_time = datetime.now()
    print(f"Total duration {(global_end_time - global_start_time).total_seconds()}s")
    print(f"Best trail: {study.best_trial.params}")

## 2.1 Load best trainer and args

In [ ]:
trainer, model_name, training_args = load_best_trainer(experiment_name=experiment_name, run_name=run_name)

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
# Don't use it uses too much memory.
# test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test")

- This uses 150GB, if doing again or much more need to write it so it does it in batches and just save the argmax.
- The actual test results are very close to the test_5_pct population, so just use that.

In [ ]:
history_pl = get_run_details(experiment_name=experiment_name, run_name=run_name, index=1)

In [ ]:
display_wide(history_pl, rows=1000)

In [ ]:
run_pl = get_run_details(experiment_name=experiment_name, run_name=run_name)

In [ ]:
run_pl.filter(pl.col("rank") == pl.col("rank").min().over("model"))

- sec-bert-base had the best f1_macro with 0.754, with train time 567s
- roberta-base second 0.743, with train time 460s - slight less performance but a bit faster and more "reputable" model
- mpnet 0.714, with train time 569s
- mini 0.681, with train time 170s - Best speed but worse performance.



# 3. Candidate model

## 3.1 1% train population

In [ ]:
trainer = train_model(model_name=model_name, training_args=training_args, train_subset="train", run_prefix="Final_model")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.2 1% sqrt weighted training population 


In [ ]:
trainer = train_model(model_name=model_name, training_args=training_args, train_subset="sample_1_pct_sqrt_weight", run_prefix="Final_model", num_train_epochs=10)

In [ ]:
model_name = "nlpaueb/sec-bert-base"

In [ ]:
trainer = load_trainer(model_name=model_name, run_prefix="Final_model", train_subset="sample_1_pct_sqrt_weight")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.3 10% sqrt training population

In [ ]:
trainer = train_model(model_name=model_name, training_args=training_args, train_subset=DataSample.sample_10_pct.label, run_prefix="Final_model")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.4 1% train population with batch size 64

In [ ]:
training_args.per_device_train_batch_size = 64
trainer = train_model(model_name=model_name, training_args=training_args, train_subset=DataSample.sample_1_pct.label, run_prefix="Final_model")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.5 1% train population with batch size 16 

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_1_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.6 1% train population with batch size 32

In [ ]:
training_args.per_device_train_batch_size = 32
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_1_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.7 10% train population with batch size 32

In [ ]:
training_args.per_device_train_batch_size = 32
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.8 100% train population with batch size 32

In [ ]:
training_args.per_device_train_batch_size = 32
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_100_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.9 1% sqrt weight train population with batch size 16

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct_sqrt_weight.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.10 10% sqrt weight train population with batch size 16

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct_sqrt_weight.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

# 4. Test Weighting model

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from datetime import datetime
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    EarlyStoppingCallback,
)
import mlflow


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


def get_class_weights(train_dataset, num_labels: int) -> torch.Tensor:
    labels = np.array(train_dataset["label"])
    present_classes = np.unique(labels)

    present_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=labels,
    )

    # full weight vector for all labels
    full_weights = np.ones(num_labels, dtype=np.float32)

    # assign computed weights only to classes present in this subset
    for cls, weight in zip(present_classes, present_weights):
        full_weights[int(cls)] = float(weight)

    return torch.tensor(full_weights, dtype=torch.float)


def train_model(
    model_name: str,
    training_args,
    train_subset: str,
    run_prefix: str,
    train_dataset: Dataset | None = None,
    num_train_epochs: int | None = None,
) -> Trainer:

    run_name = f"{run_prefix}_{train_subset}"
    output_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_name}/trial_0"

    if train_dataset is None:
        train_dataset = datasets_encoded[model_name][train_subset]
    eval_dataset = datasets_encoded[model_name]["test_5_pct"]

    class_weights = get_class_weights(train_dataset, num_labels=num_labels)

    with mlflow.start_run(run_name=run_name) as parent_run:
        with mlflow.start_run(run_name="trial_0", nested=True):
            training_args.output_dir = output_dir

            if num_train_epochs is not None:
                training_args.num_train_epochs = num_train_epochs

            # Make sure best checkpoint selection uses macro F1
            training_args.metric_for_best_model = "f1_macro"
            training_args.load_best_model_at_end = True
            training_args.greater_is_better = True

            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=num_labels,
                id2label=id2label,
                label2id=label2id,
            )

            trainer = WeightedTrainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=eval_dataset,
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
                class_weights=class_weights,
            )

            mlflow.log_params(training_args.to_dict())
            mlflow.log_param("train_subset", train_subset)

            for i, w in enumerate(class_weights.tolist()):
                mlflow.log_metric(f"class_weight_{i}", w)

            start_time = datetime.now()
            trainer.train()
            end_time = datetime.now()

            duration = (end_time - start_time).total_seconds()
            print(f"duration: {duration}s")

            mlflow.log_metric("train_time", duration)

            if trainer.state.best_metric is not None:
                mlflow.log_metric("best_metric", trainer.state.best_metric)

            trainer.save_model(output_dir)
            AutoTokenizer.from_pretrained(model_name).save_pretrained(output_dir)

    return trainer

## 4.1 1% train population

In [ ]:
training_args.per_device_train_batch_size = 16
training_args.per_device_eval_batch_size = 16
training_args.num_train_epochs = 10
training_args.metric_for_best_model = "f1_macro"
training_args.load_best_model_at_end = True
training_args.greater_is_better = True
training_args.evaluation_strategy = "epoch"
training_args.save_strategy = "epoch"
training_args.logging_strategy = "epoch"
training_args.save_total_limit = 2

trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_1_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 4.2 10% train population

In [ ]:
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=15,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="holdout_10k")

## 4.3 10% sqrt weighted train population

In [ ]:
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct_sqrt_weight.label, 
                      run_prefix="Final_model",
                      num_train_epochs=15,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="holdout_10k")

1.5 pp better than LinearSVC

# 5. Explainability

In [ ]:
dataset_pl.filter(pl.col("canonical_label").is_in(["TurnoverRevenue", "CostSales", "RawMaterialsConsumablesUsed"])).select("canonical_label", "label").unique()["label"]

In [ ]:
pipe = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=AutoTokenizer.from_pretrained(model_name),
    device=trainer.args.device,
)

explainer = shap.Explainer(pipe)
shap_values = explainer(["cost of goods sold turnover"])
top_3 = (
    dataset_pl.filter(
        pl.col("canonical_label").is_in(["TurnoverRevenue", "CostSales", "RawMaterialsConsumablesUsed"])
    )
    .select("canonical_label", "label")
    .unique()["label"]
    .to_numpy()
)
for class_idx in top_3:
    class_name = trainer.model.config.id2label[class_idx]
    print(f"Class name {class_name}")

    shap.plots.text(shap_values[:, :, class_idx])

In [ ]:
dataset_pl.filter(pl.col("label").is_in(top_3)).select("canonical_label", "label").unique()

In [ ]:
shap.plots.text(shap_values)

When you click on a label, then the contributions of the input words show up at the bottom. 
Double click to show multiple labels
This is good in that you can see the contributions to CostOfSales, so turnover has a negative contribution. But with the Turnover category it's the opposite. 

In [ ]:
tokens = shap_values.data[0]
values = shap_values.values[0]

predicted_class_idx = values.sum(axis=0).argmax()

print(f"Predicted class: {pipe.model.config.id2label[predicted_class_idx]}")

for token, value in zip(tokens, values):
    contribution_to_prediction = value[predicted_class_idx]
    max_class_idx = value.argmax()
    max_class = pipe.model.config.id2label[max_class_idx]
    print(f"{token:20s}: towards prediction={contribution_to_prediction}: strongest pull towards {max_class} - {value.max():.4f}")

This is showing what each word pulls towards, with start and stop characters as well

In [ ]:
tokens = shap_values.data[0]
values = shap_values.values[0][:,predicted_class_idx]
print(values.shape)
sns.barplot(x=tokens, y=values)

This shows how much each word contributes to the final chosen label. This looks good since you'd expect cost to be related to CostOfSales, but Turnover isn't normally linked to that concept and here it has a negative contribution which is what you'd expect. 

While transformer models aren't as transparent to what they do, the SHAP package does provide sufficient levels of explainability. 

# 6. Loss over epochs

In [ ]:
logs = trainer.state.log_history
logs_pl = pl.DataFrame(logs)
ax = sns.lineplot(logs_pl, x="epoch", y="loss", label="train_loss")
sns.lineplot(logs_pl, x="epoch", y="eval_loss", ax=ax, label="eval_loss")
plt.title("Loss over epochs")

# 7. Investigate good and bad classification 

In [ ]:
def class_confusion_matrix(y_true, y_pred, target_class):

    y_true_bin = (y_true == target_class)
    y_pred_bin = (y_pred == target_class)

    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[True, False])

    return cm

In [ ]:
model_name = trainer.model.name_or_path
prediction_details = trainer.predict(datasets_encoded[model_name]["test_5_pct"])

In [ ]:
train_pl = dataset_pl.filter(pl.col("test_5_pct"))
y_pred = prediction_details.predictions.argmax(axis=1)
y_true = train_pl["label"].to_numpy()

In [ ]:
train_pl.with_columns(pl.Series("pred_label", y_pred), pl.Series("prediction_label", le.inverse_transform(y_pred))).filter(pl.col("pred_label") != pl.col("label"))

In [ ]:
le = LabelEncoder()
le.classes_ = dataset_pl.select("label", "canonical_label").unique().sort("label")["canonical_label"]

In [ ]:
le.inverse_transform(y_pred)

In [ ]:
class_conf_matrix = class_confusion_matrix(y_true, y_pred, 127)

In [ ]:
for i in range(len(le.classes_)):
  class_matrix = class_confusion_matrix(y_true, y_pred, i)
  if(class_matrix[0][0] > 10 and class_matrix[0][1] > 1):
    print(i, le.inverse_transform([i])[0], class_matrix)

In [ ]:
def plot_confusion_matrix_heatmap(y_true, y_pred, target_class, figsize=(12, 10), normalize=False):
    cm = class_confusion_matrix(y_true, y_pred, target_class)
    print(cm)

    fig, ax = plt.subplots(figsize=figsize)

    # Create heatmap with better spacing
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Positive', 'Negative'],
                yticklabels=['Positive', 'Negative'],
                ax=ax, cbar_kws={'label': 'Count' if not normalize else 'Proportion'},
                square=True,
                annot_kws={'size': 12})

    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(le.inverse_transform([target_class])[0] + " Confusion Matrix")

    plt.xticks(rotation=0)
    plt.yticks(rotation=0)
    plt.tight_layout()

    return fig, ax



In [ ]:
plot_confusion_matrix_heatmap(y_true, y_pred, 73)
plot_confusion_matrix_heatmap(y_true, y_pred, 149)

## 7.2 Incorrect predictions

In [ ]:
import time
model.train()
batch = next(iter(DataLoader(datasets_encoded[model_name]["train"], batch_size=64)))
input_ids = batch["input_ids"].to(DEVICE)
attention_mask = batch["attention_mask"].to(DEVICE)
labels = batch["label"].to(DEVICE)

# Warmup
outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
outputs.loss.backward()

# Time it
times = []
for _ in range(20):
    start = time.time()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    outputs.loss.backward()
    times.append(time.time() - start)

print(f"mean: {sum(times)/len(times)*1000:.1f}ms/step")

# Random oversampling @TODO?


In [ ]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch: dict[str, list[Any]]) -> BatchEncoding:
    return tokenizer(batch[X], padding="max_length", truncation=True, max_length=32)


In [ ]:
df_train = dataset_pl.filter(pl.col(DataSample.sample_1_pct.label)).select(X, y).to_pandas()
# Target: min 100 samples/class, max 1000/class
target_dist = {cls: max(500, count) for cls, count in Counter(df_train[y]).items()}
ros = RandomOverSampler(sampling_strategy=target_dist, random_state=42)
X_res, y_res = ros.fit_resample(df_train[[X]], df_train[y])
print(f"Original size: {len(df_train)}") 
print(f"Resampled size: {len(X_res)}")  

train_raw = Dataset.from_pandas(pd.DataFrame({X: X_res[X], y: y_res}))

train = train_raw.map(tokenize, batched=True)
train = train.remove_columns([X])
train.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
trainer = train_model(
    model_name=model_name,
    training_args=training_args,
    train_subset=DataSample.sample_1_pct.label,
    run_prefix="Final_model",
    num_train_epochs=10,
    train_dataset=train,
)

In [ ]:
test_predictions_batch(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="holdout")